In [ ]:
!pip install statsmodels scipy -q

In [ ]:
import pandas as pd
import json

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# 수도권 전철 8호선 역별 일평균 승하차량 (출처: 나무위키, 서울교통공사)
# ※ 환승역: 양 노선 합계 승하차량
#   - 천호(5호선), 잠실(2호선), 석촌(9호선, 2018~), 가락시장(3호선, 2010~), 모란(분당선)
# ※ 석촌 2010~2017: 9호선 미개통으로 8호선 단독
# ※ 가락시장 2008~2009: 3호선 미개통으로 8호선 단독
data = {
    "별내": {2008:4858,2009:5218,2010:4215,2011:5033,2012:5646,2013:5896,2014:3428,2015:4150,2016:4651,2017:4889,2018:4858,2019:5218,2020:4215,2021:5033,2022:5646,2023:5896,2024:18568,2025:19523},
    "다산": {2024:16773,2025:18774},
    "동구릉": {2024:10582,2025:11735},
    "구리": {2008:20775,2009:22113,2010:24358,2011:25891,2012:26589,2013:27735,2014:28222,2015:28350,2016:28494,2017:28164,2018:28450,2019:29205,2020:22285,2021:22790,2022:24564,2023:25661,2024:37065,2025:33674},
    "장자호수공원": {2024:15742,2025:17082},
    "암사역사공원": {2024:10372,2025:11792},
    "암사": {2008:31364,2009:31530,2010:32402,2011:33067,2012:33399,2013:33815,2014:33989,2015:33301,2016:33191,2017:34058,2018:34938,2019:35535,2020:28773,2021:29525,2022:31576,2023:33348,2024:32695,2025:30241},
    "천호": {2008:82171,2009:82169,2010:86080,2011:88759,2012:89768,2013:90262,2014:90075,2015:86421,2016:83588,2017:82248,2018:82465,2019:80794,2020:60126,2021:58207,2022:62983,2023:67469,2024:71151,2025:75578},
    "강동구청": {2008:20665,2009:20277,2010:20657,2011:21248,2012:21563,2013:21826,2014:21823,2015:21586,2016:20943,2017:21403,2018:21790,2019:21274,2020:16880,2021:17232,2022:18402,2023:19757,2024:20186,2025:20443},
    "몽촌토성": {2008:12836,2009:12773,2010:13351,2011:13262,2012:13637,2013:13604,2014:13984,2015:14618,2016:13597,2017:14342,2018:14555,2019:12683,2020:9150,2021:10161,2022:11589,2023:12911,2024:14010,2025:14672},
    "잠실": {2008:165299,2009:168261,2010:167583,2011:170114,2012:169898,2013:171401,2014:177041,2015:182891,2016:192115,2017:202426,2018:207811,2019:205623,2020:136004,2021:137715,2022:166873,2023:185257,2024:193898,2025:199230},
    "석촌": {2008:18919,2009:18572,2010:19428,2011:19693,2012:19809,2013:19868,2014:19651,2015:18911,2016:18600,2017:17911,2018:26519,2019:29384,2020:24627,2021:26661,2022:30167,2023:33047,2024:35228,2025:36438},
    "송파": {2008:13885,2009:13548,2010:14044,2011:14188,2012:14120,2013:11657,2014:11346,2015:11157,2016:10837,2017:11018,2018:10895,2019:18081,2020:13674,2021:12649,2022:14847,2023:17237,2024:18052,2025:18615},
    "가락시장": {2008:23127,2009:22866,2010:28870,2011:30428,2012:30720,2013:31225,2014:31970,2015:31390,2016:33166,2017:35545,2018:36347,2019:36897,2020:28551,2021:27909,2022:30162,2023:32672,2024:33717,2025:34450},
    "문정": {2008:12293,2009:11563,2010:11357,2011:10787,2012:10506,2013:10589,2014:11000,2015:11560,2016:15301,2017:25571,2018:33715,2019:37833,2020:32577,2021:34218,2022:35930,2023:38152,2024:40313,2025:41696},
    "장지": {2008:9090,2009:11030,2010:15954,2011:19288,2012:20966,2013:22066,2014:24028,2015:25778,2016:28766,2017:33414,2018:36170,2019:37155,2020:27716,2021:28026,2022:30124,2023:31562,2024:31719,2025:31860},
    "복정": {2008:10676,2009:10993,2010:11326,2011:11367,2012:11128,2013:11836,2014:13660,2015:14032,2016:16905,2017:19681,2018:21210,2019:21345,2020:15636,2021:16811,2022:15642,2023:16298,2024:17132,2025:17322},
    "남위례": {2021:6303,2022:9670,2023:12934,2024:14008,2025:14394},
    "산성": {2008:14500,2009:14190,2010:14301,2011:14604,2012:14739,2013:14784,2014:15200,2015:15149,2016:14736,2017:12533,2018:11444,2019:11692,2020:9839,2021:10341,2022:9762,2023:10323,2024:12221,2025:12087},
    "남한산성입구": {2008:27261,2009:26707,2010:27181,2011:27544,2012:27674,2013:28660,2014:28975,2015:28767,2016:28021,2017:27885,2018:27612,2019:27739,2020:21041,2021:21692,2022:24224,2023:25151,2024:24895,2025:24179},
    "단대오거리": {2008:25299,2009:23866,2010:24377,2011:24522,2012:25272,2013:25440,2014:25330,2015:25303,2016:24600,2017:22777,2018:21913,2019:21782,2020:16326,2021:17271,2022:19085,2023:22690,2024:22695,2025:22678},
    "신흥": {2008:11578,2009:10823,2010:10933,2011:10727,2012:10854,2013:11087,2014:11142,2015:11185,2016:10873,2017:9985,2018:9770,2019:10163,2020:7915,2021:7922,2022:8378,2023:10169,2024:10299,2025:10458},
    "수진": {2008:10606,2009:10116,2010:10377,2011:10444,2012:10521,2013:10491,2014:10274,2015:10103,2016:10017,2017:10213,2018:10677,2019:11112,2020:8210,2021:8465,2022:9396,2023:10125,2024:10461,2025:10524},
    "모란": {2008:44984,2009:46270,2010:48170,2011:49730,2012:51544,2013:55454,2014:56861,2015:56485,2016:55903,2017:53946,2018:55101,2019:54977,2020:40772,2021:42164,2022:49353,2023:52071,2024:49835,2025:49378},
}

years = list(range(2008, 2026))
stations_order = ["별내","다산","동구릉","구리","장자호수공원","암사역사공원",
                   "암사","천호","강동구청","몽촌토성","잠실","석촌","송파","가락시장",
                   "문정","장지","복정","남위례","산성","남한산성입구","단대오거리","신흥","수진","모란"]

# DataFrame 생성 (역 = 행, 연도 = 열)
df = pd.DataFrame(index=stations_order, columns=years)
for stn in stations_order:
    for yr in years:
        val = data[stn].get(yr)
        if val is not None:
            df.loc[stn, yr] = f'{val:,}'
        else:
            df.loc[stn, yr] = '-'

df.index.name = '역명'
df.columns = [f'{y}년' for y in years]

print('=== 수도권 전철 8호선 역별 일평균 승하차량 (2008~2025) ===')
print('출처: 나무위키 (서울교통공사 자료 기반)')
print('※ 환승역: 양 노선 합계 (천호·잠실·석촌·가락시장·모란)')
print('※ 석촌 2010~2017: 9호선 미개통, 8호선 단독')
print('※ 가락시장 2008~2009: 3호선 미개통, 8호선 단독')
print()
display(df)

In [ ]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# 서울 지하철 2호선 역별 일평균 이용객 수 (출처: 나무위키, 서울교통공사 자료 기반)
# 다중 노선 역은 전 노선 합산값 (삼성역 제외)
data = {
    "시청": {2010:96999,2011:97527,2012:96122,2013:103004,2014:100083,2015:99585,2016:102613,2017:97425,2018:97422,2019:102616,2020:68708,2021:66604,2022:80407,2023:94841,2024:97755,2025:100349},
    "을지로입구": {2010:100203,2011:103388,2012:104750,2013:108352,2014:108473,2015:102628,2016:103138,2017:97151,2018:98762,2019:101199,2020:63339,2021:60024,2022:72200,2023:89277,2024:93824,2025:97173},
    "을지로3가": {2010:50469,2011:50328,2012:54419,2013:57210,2014:58357,2015:58187,2016:59207,2017:59674,2018:63739,2019:68609,2020:49273,2021:49328,2022:57633,2023:66471,2024:68923,2025:71631},
    "을지로4가": {2010:35641,2011:35221,2012:34322,2013:35012,2014:36533,2015:36578,2016:36266,2017:35098,2018:35271,2019:35928,2020:29427,2021:29517,2022:33191,2023:38051,2024:38594,2025:40255},
    "동대문역사문화공원": {2010:81045,2011:84369,2012:83531,2013:81339,2014:92815,2015:90796,2016:94234,2017:92310,2018:90508,2019:91773,2020:52932,2021:50079,2022:58479,2023:70581,2024:74613,2025:77047},
    "신당": {2010:43556,2011:44848,2012:44722,2013:46052,2014:47549,2015:47638,2016:47897,2017:48223,2018:48892,2019:48999,2020:36574,2021:37354,2022:41113,2023:45281,2024:46863,2025:46410},
    "상왕십리": {2010:18694,2011:18668,2012:19020,2013:19754,2014:20232,2015:21052,2016:21236,2017:25415,2018:27564,2019:28476,2020:22034,2021:22932,2022:24963,2023:26720,2024:27392,2025:27475},
    "왕십리": {2010:64945,2011:68142,2012:72839,2013:86348,2014:89155,2015:89166,2016:89171,2017:87386,2018:86271,2019:86352,2020:60758,2021:60194,2022:68318,2023:71734,2024:72696,2025:72672},
    "한양대": {2010:28631,2011:28003,2012:27198,2013:26319,2014:25951,2015:25830,2016:25155,2017:24449,2018:24197,2019:23819,2020:12221,2021:12225,2022:19646,2023:21626,2024:22033,2025:22978},
    "뚝섬": {2010:32736,2011:34032,2012:33093,2013:30807,2014:31764,2015:32463,2016:33404,2017:33874,2018:35609,2019:39054,2020:33145,2021:36362,2022:40701,2023:45620,2024:48800,2025:49372},
    "성수": {2010:45401,2011:48756,2012:49285,2013:50486,2014:50797,2015:51636,2016:52930,2017:54685,2018:56286,2019:60192,2020:53231,2021:58499,2022:67849,2023:78018,2024:88059,2025:102489},
    "건대입구": {2010:125915,2011:131172,2012:131055,2013:131103,2014:133713,2015:132817,2016:131767,2017:129136,2018:128112,2019:127066,2020:84200,2021:81762,2022:94833,2023:101373,2024:101982,2025:103561},
    "구의": {2010:49052,2011:49219,2012:48485,2013:48116,2014:48068,2015:48174,2016:48174,2017:46399,2018:46999,2019:47909,2020:38306,2021:38334,2022:41675,2023:45124,2024:47080,2025:50346},
    "강변": {2010:114940,2011:111070,2012:107982,2013:106194,2014:106713,2015:102791,2016:100115,2017:96202,2018:92784,2019:90150,2020:56036,2021:51220,2022:57677,2023:60234,2024:57349,2025:52821},
    "잠실나루": {2010:37702,2011:37578,2012:36826,2013:36783,2014:37237,2015:35823,2016:35317,2017:34171,2018:33347,2019:31988,2020:22355,2021:22421,2022:24677,2023:27433,2024:27728,2025:27431},
    "잠실": {2010:0,2011:170114,2012:169898,2013:171401,2014:177041,2015:182891,2016:192115,2017:202426,2018:207811,2019:205623,2020:136004,2021:137715,2022:166873,2023:185257,2024:193898,2025:199230},
    "잠실새내": {2010:0,2011:53083,2012:53513,2013:54025,2014:55148,2015:53449,2016:53452,2017:53261,2018:53257,2019:50061,2020:34170,2021:34503,2022:39613,2023:44162,2024:46004,2025:46097},
    "종합운동장": {2010:34642,2011:35223,2012:35322,2013:35067,2014:34740,2015:42799,2016:43912,2017:45095,2018:44966,2019:37279,2020:19484,2021:19901,2022:31020,2023:34255,2024:36670,2025:39210},
    "선릉": {2010:121326,2011:121790,2012:117904,2013:107826,2014:106978,2015:103360,2016:100950,2017:98916,2018:101223,2019:102302,2020:81650,2021:80005,2022:87551,2023:94347,2024:96542,2025:98117},
    "역삼": {2010:100223,2011:101767,2012:100227,2013:99258,2014:97768,2015:96908,2016:94214,2017:94231,2018:96490,2019:99387,2020:79026,2021:79587,2022:88694,2023:95440,2024:95759,2025:95588},
    "강남": {2010:203544,2011:206712,2012:207475,2013:214355,2014:213681,2015:204342,2016:199967,2017:202176,2018:204144,2019:202174,2020:142179,2021:132735,2022:142195,2023:147450,2024:149757,2025:152232},
    "교대": {2010:114452,2011:115301,2012:112237,2013:110330,2014:110463,2015:105279,2016:103944,2017:101617,2018:100547,2019:102287,2020:75186,2021:73703,2022:80830,2023:86314,2024:87233,2025:86900},
    "서초": {2010:38366,2011:39784,2012:40209,2013:40308,2014:46420,2015:46192,2016:45637,2017:44899,2018:45534,2019:46877,2020:33638,2021:33229,2022:36899,2023:40257,2024:40773,2025:41161},
    "방배": {2010:43794,2011:44576,2012:44259,2013:43910,2014:43814,2015:42808,2016:42407,2017:41560,2018:40723,2019:40100,2020:27504,2021:27651,2022:30922,2023:32632,2024:32001,2025:32620},
    "사당": {2010:150959,2011:153789,2012:154478,2013:156696,2014:158894,2015:156173,2016:151467,2017:146368,2018:145432,2019:148872,2020:104991,2021:101317,2022:118883,2023:129933,2024:133262,2025:135521},
    "낙성대": {2010:60572,2011:62170,2012:61486,2013:61771,2014:61538,2015:61314,2016:61120,2017:59794,2018:59357,2019:59167,2020:45069,2021:45065,2022:49218,2023:52060,2024:52602,2025:52988},
    "서울대입구": {2010:106358,2011:107859,2012:107536,2013:108311,2014:108563,2015:106872,2016:104992,2017:104393,2018:105330,2019:105144,2020:79393,2021:78707,2022:84873,2023:88166,2024:87928,2025:88587},
    "봉천": {2010:46173,2011:46802,2012:46826,2013:47629,2014:48373,2015:46910,2016:45637,2017:44834,2018:46202,2019:48608,2020:38797,2021:38500,2022:41245,2023:43934,2024:44817,2025:45919},
    "신림": {2010:148496,2011:147919,2012:145983,2013:146107,2014:146469,2015:146374,2016:144793,2017:139646,2018:138692,2019:139189,2020:106360,2021:104381,2022:107733,2023:104686,2024:104459,2025:106333},
    "신대방": {2010:47221,2011:50116,2012:51920,2013:53474,2014:54796,2015:55177,2016:55668,2017:55871,2018:56476,2019:56960,2020:43973,2021:43900,2022:46403,2023:48421,2024:48326,2025:47931},
    "구로디지털단지": {2010:117630,2011:122199,2012:124422,2013:125805,2014:128236,2015:127824,2016:126914,2017:124977,2018:124380,2019:126035,2020:98133,2021:96416,2022:101103,2023:106373,2024:106085,2025:106880},
    "대림": {2010:81998,2011:85860,2012:83823,2013:86710,2014:89457,2015:88275,2016:87364,2017:83759,2018:81638,2019:81233,2020:56738,2021:55847,2022:59904,2023:64980,2024:66314,2025:66201},
    "신도림": {2010:107767,2011:117425,2012:125671,2013:130389,2014:135357,2015:137770,2016:133666,2017:130258,2018:127239,2019:126407,2020:89599,2021:87356,2022:95473,2023:102499,2024:106008,2025:103845},
    "문래": {2010:38213,2011:38895,2012:38498,2013:39340,2014:38890,2015:38704,2016:37965,2017:37906,2018:39113,2019:40449,2020:33279,2021:34089,2022:37840,2023:40988,2024:43307,2025:44474},
    "영등포구청": {2010:48390,2011:49740,2012:50261,2013:50109,2014:50614,2015:50484,2016:50286,2017:50845,2018:50845,2019:52092,2020:43007,2021:43550,2022:45871,2023:48800,2024:51426,2025:53126},
    "당산": {2010:73572,2011:77618,2012:77214,2013:78471,2014:80445,2015:80086,2016:80345,2017:79916,2018:80076,2019:83903,2020:62524,2021:61808,2022:68409,2023:73730,2024:73808,2025:73973},
    "합정": {2010:66804,2011:69745,2012:73626,2013:85372,2014:94386,2015:93815,2016:94221,2017:96564,2018:98658,2019:100127,2020:74105,2021:74700,2022:85659,2023:93434,2024:94017,2025:93373},
    "홍대입구": {2010:0,2011:135157,2012:147444,2013:158703,2014:175065,2015:184724,2016:187853,2017:193413,2018:198473,2019:205323,2020:117806,2021:116934,2022:150023,2023:178929,2024:185474,2025:189503},
    "신촌": {2010:112799,2011:114035,2012:112116,2013:110448,2014:109748,2015:104854,2016:103971,2017:100228,2018:98409,2019:95972,2020:58089,2021:54995,2022:65740,2023:74872,2024:75866,2025:75597},
    "이대": {2010:49070,2011:49178,2012:48874,2013:49215,2014:50994,2015:47450,2016:45332,2017:41576,2018:41143,2019:41008,2020:20808,2021:21366,2022:27009,2023:32452,2024:33708,2025:34331},
    "아현": {2010:21496,2011:18547,2012:16725,2013:16528,2014:17227,2015:19103,2016:20104,2017:20296,2018:21055,2019:21477,2020:15848,2021:16910,2022:18423,2023:20069,2024:20414,2025:20616},
    "충정로": {2010:32607,2011:32583,2012:34988,2013:36106,2014:34922,2015:33511,2016:33763,2017:32952,2018:32152,2019:31686,2020:23982,2021:23702,2022:25474,2023:27582,2024:28241,2025:28851},
    "용답": {2010:5471,2011:5743,2012:5789,2013:5517,2014:5481,2015:5365,2016:5358,2017:5635,2018:5811,2019:5876,2020:4498,2021:4832,2022:5667,2023:5923,2024:6045,2025:6090},
    "신답": {2010:3393,2011:3373,2012:3186,2013:2961,2014:2952,2015:2900,2016:2919,2017:2949,2018:3203,2019:3476,2020:2770,2021:2865,2022:3189,2023:3349,2024:3325,2025:3373},
    "용두": {2010:4569,2011:4656,2012:4662,2013:4637,2014:4696,2015:4600,2016:4725,2017:4748,2018:4905,2019:4996,2020:3930,2021:4068,2022:4352,2023:4718,2024:4949,2025:5062},
    "신설동": {2010:43077,2011:42881,2012:41849,2013:41018,2014:41935,2015:41314,2016:40570,2017:42502,2018:43933,2019:44546,2020:32656,2021:32138,2022:34983,2023:37660,2024:38975,2025:39558},
    "도림천": {2010:1667,2011:2011,2012:2035,2013:1966,2014:2049,2015:2105,2016:2245,2017:2301,2018:2471,2019:2527,2020:2119,2021:2329,2022:2460,2023:2498,2024:2615,2025:2680},
    "양천구청": {2010:16833,2011:16639,2012:16314,2013:16196,2014:15694,2015:15301,2016:14944,2017:14327,2018:14192,2019:14982,2020:11195,2021:11531,2022:12562,2023:13545,2024:13503,2025:13445},
    "신정네거리": {2010:24744,2011:24205,2012:23388,2013:22959,2014:22587,2015:22118,2016:21460,2017:19875,2018:19516,2019:20470,2020:15999,2021:17329,2022:18816,2023:19948,2024:19940,2025:19953},
    "까치산": {2010:55879,2011:56234,2012:57105,2013:57873,2014:58951,2015:58780,2016:58405,2017:58515,2018:58777,2019:59394,2020:47311,2021:47637,2022:50656,2023:53017,2024:52463,2025:52221},
}

years = list(range(2010, 2026))
stations_order = list(data.keys())

df = pd.DataFrame(index=stations_order, columns=years)
for stn in stations_order:
    for yr in years:
        val = data[stn].get(yr, 0)
        df.loc[stn, yr] = f'{val:,}' if val > 0 else '0'

df.index.name = '역명'
df.columns = [f'{y}년' for y in years]

print('=== 서울 지하철 2호선 역별 일평균 이용객 수 (2010~2025) ===')
print('출처: 나무위키 (서울교통공사 자료 기반)')
print('※ 다중 노선 역은 전 노선 합산값, 삼성역 제외')
print()
display(df)

In [ ]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# 수도권 전철 8호선 역간 거리 및 간섭지수
# L = 1 - e^(-2.77x²), x = 인접 역간 거리 평균(km)
df_L = pd.read_csv(
    "수도권전철8호선_역번호_거리_역명.csv",
    encoding="utf-8-sig"
)

print("=== 수도권 전철 8호선 역간 거리 및 간섭지수(L) ===")
print()
display(df_L)

# 8호선 역세권 읍면동별 연령대별 야간인구밀도 (2011~2025)
출처: 통계청 주민등록인구 5세별  
- 0~14세 / 15~64세 / 65세 이상으로 재분류  
- **주간인구밀도 = 상주인구(주민등록인구)와 동일하다고 가정**

In [ ]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# CSV 읽기
df_pop = pd.read_csv(
    "101_DT_1B04005N_20260408025031.csv",
    encoding="euc-kr"
)
df_pop.columns = ["행정구역","연령","항목","단위"] + [f"{y}" for y in range(2011, 2026)] + ["_"]
df_pop = df_pop.drop(columns=["항목","단위","_"], errors="ignore")

# 구·시 단위 제외, 동 단위만
gu_si = ["강동구","송파구","수정구","중원구","구리시","남양주시"]
df_pop = df_pop[~df_pop["행정구역"].isin(gu_si)].copy()

# 연도 컬럼 숫자 변환
year_cols = [str(y) for y in range(2011, 2026)]
for c in year_cols:
    df_pop[c] = pd.to_numeric(df_pop[c], errors="coerce").fillna(0).astype(int)

# 연령대 재분류: 0~14, 15~64, 65+
age_0_14 = ["0 - 4세", "5 - 9세", "10 - 14세"]
age_15_64 = ["15 - 19세","20 - 24세","25 - 29세","30 - 34세","35 - 39세",
             "40 - 44세","45 - 49세","50 - 54세","55 - 59세","60 - 64세"]
age_65p = ["65 - 69세","70 - 74세","75 - 79세","80 - 84세","85 - 89세",
           "90 - 94세","95 - 99세","100+"]

def agg_age(df, age_list, label):
    sub = df[df["연령"].isin(age_list)].groupby("행정구역")[year_cols].sum().reset_index()
    sub.insert(1, "연령대", label)
    return sub

df_0_14 = agg_age(df_pop, age_0_14, "0~14세")
df_15_64 = agg_age(df_pop, age_15_64, "15~64세")
df_65p = agg_age(df_pop, age_65p, "65세 이상")
df_total = df_pop[df_pop["연령"] == "계"].copy()
df_total["연령대"] = "계"
df_total = df_total[["행정구역","연령대"] + year_cols]

df_age = pd.concat([df_total, df_0_14, df_15_64, df_65p], ignore_index=True)
df_age = df_age.sort_values(["행정구역","연령대"]).reset_index(drop=True)

# 역 ↔ 행정동 매핑 (역사 소재 행정동, 경계 시 모두 포함)
station_dong = {
    "별내": ["별내동"],
    "다산": ["다산1동"],
    "동구릉": ["동구동"],
    "구리": ["인창동"],
    "장자호수공원": ["수택3동", "교문2동"],
    "암사역사공원": ["암사제1동", "암사제3동"],
    "암사": ["암사제1동", "암사제2동"],
    "천호": ["천호제2동", "성내제2동", "풍납1동"],
    "강동구청": ["성내제1동", "풍납2동"],
    "몽촌토성": ["방이2동", "잠실4동"],
    "잠실": ["잠실6동", "잠실4동"],
    "석촌": ["석촌동", "송파1동"],
    "송파": ["송파2동", "가락1동"],
    "가락시장": ["가락1동", "문정2동"],
    "문정": ["문정1동", "문정2동"],
    "장지": ["장지동"],
    "복정": ["장지동", "복정동"],
    "남위례": ["복정동"],
    "산성": ["산성동", "신흥2동"],
    "남한산성입구": ["단대동", "금광2동"],
    "단대오거리": ["단대동", "신흥2동", "금광1동", "금광2동", "중앙동"],
    "신흥": ["신흥3동", "중앙동"],
    "수진": ["수진1동", "중앙동"],
    "모란": ["성남동", "수진2동"],
}

# 역별 연령대별 인구 집계
rows = []
for stn, dongs in station_dong.items():
    for age_label in ["계","0~14세","15~64세","65세 이상"]:
        sub = df_age[(df_age["행정구역"].isin(dongs)) & (df_age["연령대"] == age_label)]
        vals = sub[year_cols].sum()
        row = {"역명": stn, "연령대": age_label}
        row.update(vals.to_dict())
        rows.append(row)

df_station_pop = pd.DataFrame(rows)

# 피벗: 역별 연도별 표시
stations_order = ["별내","다산","동구릉","구리","장자호수공원","암사역사공원",
                   "암사","천호","강동구청","몽촌토성","잠실","석촌","송파","가락시장",
                   "문정","장지","복정","남위례","산성","남한산성입구","단대오거리","신흥","수진","모란"]

for age_label in ["계","0~14세","15~64세","65세 이상"]:
    print(f"\n=== 역세권 {age_label} 야간인구밀도 ===")
    sub = df_station_pop[df_station_pop["연령대"] == age_label].set_index("역명")[year_cols]
    sub = sub.reindex([s for s in stations_order if s in sub.index])
    display(sub)

In [ ]:
import pandas as pd

# 8호선 역세권 행정동 면적 (km²)
# 출처: 송파통계연보(2025), 강동통계연보(2025), 성남시 행정구역 CSV, 구리시 행정구역 CSV, 남양주시 행정구역 CSV
dong_area = {
    # 남양주시
    "별내동": 18.64, "다산1동": 5.55,
    # 구리시
    "동구동": 7.32, "인창동": 2.10, "교문1동": 7.55, "교문2동": 1.15,
    "수택1동": 1.26, "수택3동": 9.34,
    # 강동구
    "암사제1동": 1.02, "암사제2동": 1.18, "암사제3동": 2.51,
    "천호제2동": 1.57, "천호제3동": 0.79,
    "성내제1동": 0.58, "성내제2동": 0.67,
    "풍납1동": 0.77, "풍납2동": 1.59,
    # 송파구
    "방이2동": 0.80, "잠실4동": 1.56, "잠실6동": 1.37,
    "석촌동": 0.95, "송파1동": 0.79,
    "송파2동": 0.53, "가락본동": 1.13,
    "가락1동": 1.34, "문정1동": 0.56, "문정2동": 2.20,
    "장지동": 2.79, "복정동": 1.50,
    # 성남시 수정구
    "산성동": 0.59, "신흥1동": 0.39, "신흥2동": 1.02, "신흥3동": 3.0,
    "단대동": 0.80, "수진1동": 0.39, "수진2동": 0.86,
    # 성남시 중원구
    "금광1동": 0.72, "금광2동": 1.01,
    "중앙동": 0.68, "성남동": 1.99, "은행2동": 2.54,
}

# 역별 면적 + 인구 합산
station_dong = {
    "별내": ["별내동"],
    "다산": ["다산1동"],
    "동구릉": ["동구동"],
    "구리": ["인창동"],
    "장자호수공원": ["수택3동", "교문2동"],
    "암사역사공원": ["암사제1동", "암사제3동"],
    "암사": ["암사제1동", "암사제2동"],
    "천호": ["천호제2동", "성내제2동", "풍납1동"],
    "강동구청": ["성내제1동", "풍납2동"],
    "몽촌토성": ["방이2동", "잠실4동"],
    "잠실": ["잠실6동", "잠실4동"],
    "석촌": ["석촌동", "송파1동"],
    "송파": ["송파2동", "가락1동"],
    "가락시장": ["가락1동", "문정2동"],
    "문정": ["문정1동", "문정2동"],
    "장지": ["장지동"],
    "복정": ["장지동", "복정동"],
    "남위례": ["복정동"],
    "산성": ["산성동", "신흥2동"],
    "남한산성입구": ["단대동", "금광2동"],
    "단대오거리": ["단대동", "신흥2동", "금광1동", "금광2동", "중앙동"],
    "신흥": ["신흥3동", "중앙동"],
    "수진": ["수진1동", "중앙동"],
    "모란": ["성남동", "수진2동"],
}

# df_station_pop은 이전 셀에서 생성된 변수
year_cols = [str(y) for y in range(2011, 2026)]
pop_total = df_station_pop[df_station_pop["연령대"] == "계"].set_index("역명")[year_cols]

rows = []
for stn, dongs in station_dong.items():
    area = sum(dong_area.get(d, 0) for d in dongs)
    row = {"역명": stn, "소재 행정동": ", ".join(dongs), "면적합(km²)": round(area, 2)}
    if stn in pop_total.index:
        for y in year_cols:
            row[f"인구({y})"] = int(pop_total.loc[stn, y])
    rows.append(row)

df_area_pop = pd.DataFrame(rows)
df_area_pop.index = df_area_pop["역명"]
df_area_pop.index.name = None

# 면적 + 인구(대표 연도) 표시
display_cols = ["소재 행정동","면적합(km²)"] + [f"인구({y})" for y in ["2011","2015","2020","2025"]]
print("=== 8호선 역세권 행정동 면적 및 인구 ===")
print()
display(df_area_pop[display_cols])

# 전 연도 인구 표
print()
print("=== 역세권 총 인구 (2011~2025) ===")
pop_cols = [f"인구({y})" for y in year_cols]
display(df_area_pop[["면적합(km²)"] + pop_cols])

In [ ]:
# 역세권 연도별 연령대별 인구밀도 (명/km²) = 연령대별 인구 / 면적
year_cols = [str(y) for y in range(2011, 2026)]

stations_order = ["별내","다산","동구릉","구리","장자호수공원","암사역사공원",
                   "암사","천호","강동구청","몽촌토성","잠실","석촌","송파","가락시장",
                   "문정","장지","복정","남위례","산성","남한산성입구","단대오거리","신흥","수진","모란"]

area_series = df_area_pop.set_index("역명")["면적합(km²)"]

for age_label in ["0~14세", "15~64세", "65세 이상"]:
    pop_sub = df_station_pop[df_station_pop["연령대"] == age_label].set_index("역명")[year_cols]
    pop_sub = pop_sub.reindex([s for s in stations_order if s in pop_sub.index])
    
    density = pop_sub.div(area_series, axis=0).round(0).astype(int)
    
    print(f"\n=== 역세권 {age_label} 인구밀도 (명/km²) ===")
    display(density)

# 8호선 역세권 아파트 / 비아파트 가구 수 (2020년 인구총조사)
출처: 통계청 거처의 종류별 가구 (읍면동)  
- 아파트 가구 수 = 주택_아파트  
- 비아파트 가구 수 = 일반가구 - 아파트 가구 수

In [ ]:
import pandas as pd

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# CSV 읽기
df_h = pd.read_csv(
    "거처의_종류별_가구__읍면동_연도_끝자리_0__5___시군구_그_외_연도__20260408033530.csv",
    header=None, skiprows=2, encoding="utf-8-sig"
)
df_h.columns = ["행정구역","일반가구","주택계","단독주택","아파트","연립주택","다세대주택","비주거용건물내주택","주택이외거처"]

# 구·시 단위 제외
gu_si = ["송파구","강동구","수정구","중원구","구리시","남양주시"]
df_h = df_h[~df_h["행정구역"].isin(gu_si)].copy()

# X → 0, 숫자 변환
for c in ["일반가구","아파트"]:
    df_h[c] = pd.to_numeric(df_h[c], errors="coerce").fillna(0).astype(int)

# 비아파트 = 일반가구 - 아파트
df_h["비아파트"] = df_h["일반가구"] - df_h["아파트"]

# 역별 집계
station_dong = {
    "별내": ["별내동"],
    "다산": ["다산1동"],
    "동구릉": ["동구동"],
    "구리": ["인창동"],
    "장자호수공원": ["수택3동", "교문2동"],
    "암사역사공원": ["암사제1동", "암사제3동"],
    "암사": ["암사제1동", "암사제2동"],
    "천호": ["천호제2동", "성내제2동", "풍납1동"],
    "강동구청": ["성내제1동", "풍납2동"],
    "몽촌토성": ["방이2동", "잠실4동"],
    "잠실": ["잠실6동", "잠실4동"],
    "석촌": ["석촌동", "송파1동"],
    "송파": ["송파2동", "가락1동"],
    "가락시장": ["가락1동", "문정2동"],
    "문정": ["문정1동", "문정2동"],
    "장지": ["장지동"],
    "복정": ["장지동", "복정동"],
    "남위례": ["복정동"],
    "산성": ["산성동", "신흥2동"],
    "남한산성입구": ["단대동", "금광2동"],
    "단대오거리": ["단대동", "신흥2동", "금광1동", "금광2동", "중앙동"],
    "신흥": ["신흥3동", "중앙동"],
    "수진": ["수진1동", "중앙동"],
    "모란": ["성남동", "수진2동"],
}

rows = []
for stn, dongs in station_dong.items():
    sub = df_h[df_h["행정구역"].isin(dongs)]
    apt = int(sub["아파트"].sum())
    non_apt = int(sub["비아파트"].sum())
    total = apt + non_apt
    pct = round(apt / total * 100, 1) if total > 0 else 0
    rows.append({"역명": stn, "아파트": apt, "비아파트": non_apt, "합계": total, "아파트비율(%)": pct})

df_housing = pd.DataFrame(rows).set_index("역명")

print("=== 8호선 역세권 아파트 / 비아파트 가구 수 (2020년) ===")
print()
display(df_housing)

In [ ]:
# 8호선 수송력 상수 K
# 6량 편성, 1량 정원 160명, 평일 운행 320회 (가락시장역 기준, 8902 회송·8321 가락시장종착 제외)
cars = 6
capacity_per_car = 160
daily_runs = 320

K_line8 = cars * capacity_per_car * daily_runs

print(f"=== 8호선 수송력 상수 K ===")
print(f"편성: {cars}량, 1량 정원: {capacity_per_car}명, 평일 운행: {daily_runs}회")
print(f"K = {cars} × {capacity_per_car} × {daily_runs} = {K_line8:,}명")

# 회귀분석 모형

## 모형 1: 이용객 수 예측 (기존)
$$\ln(y_{i,t}) = eta_0 + \sum_j eta_j X_{j,i,t} + eta_k \ln(K_i) + eta_l \ln(L_i) + \delta_t + \epsilon_{i,t}$$

## 모형 2: 주간인구밀도 추정 (역모형)
$$\ln(x_{36,i,t}) = \gamma_0 + \gamma_y \ln(y_{i,t}) + \sum_m \gamma_m Z_{m,i,t} + \gamma_k \ln(K_i) + \gamma_l \ln(L_i) + \delta_t + \eta_{i,t}$$

- $y_{i,t}$: 역 $i$의 연도 $t$ 일평균 승하차량
- $x_{36,i,t}$: 역 $i$ 영향권 내 주간 활동 인구 (**추정 대상**)
- $X_j, Z_m$: 인구밀도(0~14세, 15~64세, 65세+), 가구밀도 등 통제 변수
- $K_i$: 수송력 상수, $L_i$: 역간 거리 간섭지수
- $\delta_t$: 연도 고정효과, $\gamma_y$: 주간인구밀도의 이용객 탄력성
- 주간인구밀도 = 상주인구 가정을 해제하고, 모형 1의 결과를 활용하여 역별 주간인구밀도를 역추정

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# ══════════════════════════════════════════════════════════════
# 승법적 모형 (로그 변환, 인구밀도 사용, β_K 자유추정)
#
# 원형: y = β₀ · exp(β₂₃x₂₃ + β₂₄x₂₄ + β₂₅x₂₅) · K^βK · L^βL · exp(δt) · ε
# 변환: ln(y) = ln(β₀) + β₂₃x₂₃ + β₂₄x₂₄ + β₂₅x₂₅ + βK·ln(K) + βL·ln(L) + δt + ln(ε)
#
# x₂₃, x₂₄, x₂₅ = 인구밀도(명/km²) — 균등분포 가정
# 주간인구밀도 추정: x₃₆ = x_night × exp(ε̂)
# ══════════════════════════════════════════════════════════════

stations_order = ["별내","다산","동구릉","구리","장자호수공원","암사역사공원",
                   "암사","천호","강동구청","몽촌토성","잠실","석촌","송파","가락시장",
                   "문정","장지","복정","남위례","산성","남한산성입구","단대오거리","신흥","수진","모란"]
year_cols = [str(y) for y in range(2011, 2026)]

K_base = 307200  # 6량 × 160명 × 320회
K_mult = {"별내":1.4, "구리":1.6, "천호":3.0, "잠실":5.0,
           "가락시장":2.6, "석촌":2.5, "복정":2.1, "모란":2.1}

# ── 패널 구축 (인구밀도 = 인구/면적, 캡 없음) ──
# 간섭지수 L = 1 - exp(-2.77 * x^2), x = 인접역간 평균거리(km)
import math
_calc_L = lambda x: 1 - math.exp(-2.77 * x**2)
L_map = {
    "별내": _calc_L(3.50), "다산": _calc_L(2.40), "동구릉": _calc_L(1.55),
    "구리": _calc_L(1.50), "장자호수공원": _calc_L(2.80), "암사역사공원": _calc_L(1.20),
    "암사": _calc_L(1.05), "천호": _calc_L(1.10), "강동구청": _calc_L(1.15),
    "몽촌토성": _calc_L(1.10), "잠실": _calc_L(1.00), "석촌": _calc_L(0.85),
    "송파": _calc_L(0.70), "가락시장": _calc_L(0.75), "문정": _calc_L(0.90),
    "장지": _calc_L(0.90), "복정": _calc_L(1.20), "남위례": _calc_L(1.00),
    "산성": _calc_L(1.25), "남한산성입구": _calc_L(1.05), "단대오거리": _calc_L(0.70),
    "신흥": _calc_L(0.65), "수진": _calc_L(0.60), "모란": _calc_L(2.50),
}

rows = []
for stn, dongs in station_dong.items():
    p23 = df_age[(df_age["행정구역"].isin(dongs)) & (df_age["연령대"]=="0~14세")][year_cols].sum()
    p24 = df_age[(df_age["행정구역"].isin(dongs)) & (df_age["연령대"]=="15~64세")][year_cols].sum()
    p25 = df_age[(df_age["행정구역"].isin(dongs)) & (df_age["연령대"]=="65세 이상")][year_cols].sum()
    p_total = df_age[(df_age["행정구역"].isin(dongs)) & (df_age["연령대"]=="계")][year_cols].sum()

    admin_area = sum(dong_area.get(d, 0) for d in dongs)
    L = L_map.get(stn, np.nan)
    K = K_base * K_mult.get(stn, 1.0)

    for yr in range(2011, 2026):
        y_val = data.get(stn, {}).get(yr)
        if y_val is None or y_val == 0: continue
        v23, v24, v25 = int(p23[str(yr)]), int(p24[str(yr)]), int(p25[str(yr)])
        vt = int(p_total[str(yr)])
        if v23 == 0 and v24 == 0 and v25 == 0: continue
        rows.append({"역명":stn, "연도":yr, "y":y_val,
            "x23":v23/admin_area, "x24":v24/admin_area, "x25":v25/admin_area,
            "nighttime_pop":vt, "nighttime_density":vt/admin_area,
            "admin_area":admin_area,
            "ln_K":np.log(K), "ln_L":np.log(L) if L and L > 0 else np.nan})

panel = pd.DataFrame(rows).dropna()
panel = panel[(panel["x23"]>0) & (panel["x24"]>0) & (panel["x25"]>0)]
panel["ln_y"] = np.log(panel["y"])

# ── OLS 회귀 ──
year_dummies = pd.get_dummies(panel["연도"], prefix="Y", drop_first=True, dtype=float)
X = sm.add_constant(pd.concat([panel[["x23","x24","x25","ln_K","ln_L"]], year_dummies], axis=1))
model = sm.OLS(panel["ln_y"], X).fit(cov_type='HC1')

print("=" * 90)
print("승법적 모형 (인구밀도/km², β_K 자유추정, 잠실 K=5.0)")
print("=" * 90)
print(model.summary())

print("\n=== 주요 계수 ===")
for var in ["x23","x24","x25","ln_K","ln_L"]:
    c, p = model.params[var], model.pvalues[var]
    sig = "***" if p<0.01 else "**" if p<0.05 else "*" if p<0.1 else ""
    print(f"  {var:>6}: β={c:+.6f} (p={p:.4f}) {sig}")
print(f"\n  R²={model.rsquared:.4f}, Adj.R²={model.rsquared_adj:.4f}, N={int(model.nobs)}")
print(f"  βK={model.params['ln_K']:.4f} → K 1%↑ → y {model.params['ln_K']:.2f}%↑")

# ── 주간인구밀도 추정 ──
panel["ln_y_pred"] = model.predict(X)
panel["residual"] = panel["ln_y"] - panel["ln_y_pred"]
panel["daytime_ratio"] = np.exp(panel["residual"])
panel["daytime_density"] = panel["nighttime_density"] * panel["daytime_ratio"]

# 2025년 결과 (내림차순)
print(f"\n{'='*115}")
print("주간인구밀도 추정 (exp(ε̂) 내림차순)")
print(f"{'='*115}")
print(f"{'역명':<12} {'면적':>6} {'야간인구밀도':>10} {'이용객':>8} {'예측이용객':>10} {'exp(ε̂)':>8} {'주간인구밀도':>12}")
print("-" * 115)

result_rows = []
rows = []
for stn in stations_order:
    p = panel[(panel["역명"]==stn) & (panel["연도"].isin([2025,2024]))]
    if len(p) == 0: continue
    result_rows.append(p.iloc[-1])

result_rows.sort(key=lambda r: r["daytime_ratio"], reverse=True)
for r in result_rows:
    y_pred = np.exp(r["ln_y_pred"])
    print(f"{r['역명']:<12} {r['admin_area']:>6.2f} {r['nighttime_density']:>10,.0f} {r['y']:>8,} {y_pred:>10,.0f} {r['daytime_ratio']:>8.2f} {r['daytime_density']:>12,.0f}")

# 전 연도 피벗
print(f"\n{'='*115}")
print("역별 연도별 추정 주간인구밀도")
print(f"{'='*115}")
pivot = panel.pivot_table(index="역명", columns="연도", values="daytime_density", aggfunc="first")
pivot = pivot.reindex([s for s in stations_order if s in pivot.index])
for col in pivot.columns:
    pivot[col] = pivot[col].apply(lambda x: f"{x:,.0f}" if pd.notna(x) else "-")
display(pivot)

In [ ]:
import numpy as np
from scipy.stats import t as t_dist

# ══════════════════════════════════════════════════════════════
# 역 고정효과(FE) 모형 — Within Estimator
# ln(K), ln(L)은 시불변 → 역별 고정효과(α_i)에 흡수
# ══════════════════════════════════════════════════════════════

tv = ["ln_y","x23","x24","x25"] + [f"Y_{yr}" for yr in range(2012,2026)]
dm = panel.copy()
for v in tv:
    dm[v+"_dm"] = dm.groupby("역명")[v].transform(lambda x: x - x.mean())

feat_fe = ["x23_dm","x24_dm","x25_dm"] + [f"Y_{yr}_dm" for yr in range(2012,2026)]
Xf = dm[feat_fe].values
yf = dm["ln_y_dm"].values
ns = panel["역명"].nunique()
nf = len(yf)
kf = Xf.shape[1]
dfe = nf - ns - kf

bf = np.linalg.inv(Xf.T @ Xf) @ Xf.T @ yf
rf = yf - Xf @ bf
s2f = (rf @ rf) / dfe
sef = np.sqrt(np.diag(s2f * np.linalg.inv(Xf.T @ Xf)))
tfv = bf / sef
pf = 2 * (1 - t_dist.cdf(np.abs(tfv), df=dfe))
DWf = np.sum(np.diff(rf)**2) / np.sum(rf**2)

fvn = ["x23","x24","x25"] + [f"Y_{yr}" for yr in range(2012,2026)]
alpha_i = {}
for stn in panel["역명"].unique():
    m = panel["역명"] == stn
    alpha_i[stn] = panel.loc[m,"ln_y"].mean() - panel.loc[m,fvn].mean().values @ bf

panel["alpha_i"] = panel["역명"].map(alpha_i)
panel["ln_y_fe"] = panel["alpha_i"]
for j, v in enumerate(fvn):
    panel["ln_y_fe"] += bf[j] * panel[v]

rfo = panel["ln_y"].values - panel["ln_y_fe"].values
yo = panel["ln_y"].values
R2fe = 1 - np.sum(rfo**2) / np.sum((yo - yo.mean())**2)

panel["ratio_fe"] = np.exp(rfo)
panel["day_fe_density"] = panel["nighttime_density"] * panel["ratio_fe"]
panel["ratio_ols"] = panel["daytime_ratio"]

# 출력
print("=" * 80)
print("Pooled OLS vs 역 고정효과(FE)")
print("=" * 80)
r2_ols = 1 - np.sum(residuals**2) / np.sum((yo - yo.mean())**2)
dw_ols = np.sum(np.diff(residuals)**2) / np.sum(residuals**2)
print(f"{'':>20} {'Pooled OLS':>15} {'역 FE':>15}")
print("-" * 55)
print(f"{'R2':>20} {r2_ols:>15.4f} {R2fe:>15.4f}")
print(f"{'Durbin-Watson':>20} {dw_ols:>15.3f} {DWf:>15.3f}")

print(f"\n{'변수':<12} {'OLS β':>12} {'FE β':>12} {'FE p':>10}")
print("-" * 50)
col_names_ols = list(X_df.columns)
for v in ["x23","x24","x25"]:
    oi = col_names_ols.index(v); fi = fvn.index(v)
    sig = "***" if pf[fi]<0.01 else "**" if pf[fi]<0.05 else "*" if pf[fi]<0.1 else ""
    print(f"  {v:<10} {beta[oi]:>+12.6f} {bf[fi]:>+12.6f} {pf[fi]:>10.4f} {sig}")
print(f"  {'ln_K':<10} {beta[col_names_ols.index('ln_K')]:>+12.6f} {'(흡수)':>12}")
print(f"  {'ln_L':<10} {beta[col_names_ols.index('ln_L')]:>+12.6f} {'(흡수)':>12}")

print(f"\n{'='*115}")
print("주간인구밀도 추정 비교: OLS vs FE (FE exp(ε̂) 내림차순)")
print(f"{'='*115}")
print(f"{'역명':<12} {'α_i':>8} {'야간인구밀도':>10} {'이용객':>8} {'OLS비율':>8} {'FE비율':>8} {'OLS주간밀도':>12} {'FE주간밀도':>12}")
print("-" * 115)

result_rows = []
for stn in stations_order:
    p = panel[(panel["역명"]==stn) & (panel["연도"].isin([2025,2024]))]
    if len(p) == 0: continue
    result_rows.append(p.iloc[-1])
result_rows.sort(key=lambda r: r["ratio_fe"], reverse=True)

for r in result_rows:
    ols_day_density = r["nighttime_density"] * r["ratio_ols"]
    print(f"{r['역명']:<12} {alpha_i[r['역명']]:>8.3f} {r['nighttime_density']:>10,.0f} {r['y']:>8,} {r['ratio_ols']:>8.2f} {r['ratio_fe']:>8.2f} {ols_day_density:>12,.0f} {r['day_fe_density']:>12,.0f}")